<a href="https://colab.research.google.com/github/Metropoliya/AI/blob/main/fb2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q edge-tts beautifulsoup4 lxml natsort

In [ ]:
import os
import re
import asyncio
import textwrap
import edge_tts
from bs4 import BeautifulSoup
from natsort import natsorted
from google.colab import drive
from google.colab import files

# 1. Подключаем Google Диск (файлы будут сохраняться напрямую туда)
drive.mount('/content/drive')

# --- НАСТРОЙКИ ---
VOICE = "ru-RU-SvetlanaNeural"

# Папка, куда ты закинул свои .fb2 файлы (временная память Colab)
INPUT_DIR = '/content'

# Главная папка, куда будут сохраняться готовые книги и фрагменты на Диске
SAVE_BASE_DIR = '/content/drive/MyDrive/Astrology_Audiobooks'

os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(SAVE_BASE_DIR, exist_ok=True)

print(f"📁 Исходные книги (fb2) ищем в: {INPUT_DIR}")
print(f"📁 Готовые аудиокниги будут тут: {SAVE_BASE_DIR}")
print("-" * 40)

# --- ФУНКЦИИ ---

def get_text_from_fb2(file_path):
    """Извлекает чистый текст из файла формата FB2"""
    # Читаем в режиме 'rb' (байты), чтобы парсер сам определил кодировку (защита от кракозябр)
    with open(file_path, 'rb') as f:
        soup = BeautifulSoup(f, 'xml')

    # Собираем все абзацы текста
    paragraphs = [p.text.strip() for p in soup.find_all('p') if p.text]
    full_text = " ".join(paragraphs)

    # Убираем лишние пробелы и переносы
    full_text = re.sub(r'\s+', ' ', full_text)
    return full_text

def chunk_text(text, max_chars=1500):
    """Разбивает текст на куски (лимит 1500 символов, чтобы сервер edge-tts не зависал)"""
    return textwrap.wrap(text, width=max_chars, break_long_words=False)

async def generate_bulletproof_audio(text_chunks, book_output_dir):
    """Асинхронная генерация аудио с защитой от обрывов Colab и подробным логом"""
    print(f"Всего фрагментов: {len(text_chunks)}")

    for i, chunk in enumerate(text_chunks):
        # Формируем имя файла с ведущими нулями для правильной сортировки (chunk_0000.mp3)
        file_name = f"chunk_{i:04d}.mp3"
        file_path = os.path.join(book_output_dir, file_name)

        # ❗ ЗАЩИТА: Если файл уже есть, пропускаем
        if os.path.exists(file_path):
            print(f"⏭️ Фрагмент {i + 1} уже существует. Пропускаем...")
            continue

        print(f"🎙️ Озвучиваем фрагмент {i + 1} / {len(text_chunks)}... (символов: {len(chunk)})")

        retries = 3
        while retries > 0:
            try:
                communicate = edge_tts.Communicate(chunk, VOICE)
                await communicate.save(file_path)

                print(f"✅ Готово!")

                await asyncio.sleep(1.0) # Вежливая пауза
                break # Выходим из цикла попыток при успехе

            except Exception as e:
                print(f"⚠️ Сбой сети на куске {i + 1}. Осталось попыток: {retries-1}. Ошибка: {e}")
                retries -= 1
                await asyncio.sleep(3)

def merge_audio_for_book(book_output_dir, final_audio_path):
    """Склеивает все фрагменты одной книги в единый mp3 файл"""
    print(f"Подготавливаем магию склейки для книги...")

    chunks = [f for f in os.listdir(book_output_dir) if f.startswith("chunk_") and f.endswith(".mp3")]
    chunks.sort()

    if not chunks:
        print("Файлы для склейки не найдены!")
        return False

    list_file_path = os.path.join(book_output_dir, "file_list.txt")
    with open(list_file_path, "w") as f:
        for chunk in chunks:
            chunk_path = os.path.join(book_output_dir, chunk)
            f.write(f"file '{chunk_path}'\n")

    print("Склеиваем файлы воедино (ffmpeg)...")
    # -c copy обеспечивает мгновенную склейку без перекодирования
    os.system(f"ffmpeg -f concat -safe 0 -i '{list_file_path}' -c copy '{final_audio_path}' -y")
    return True

# --- ГЛАВНЫЙ ЦИКЛ (ЗАПУСК) ---
async def process_all_books():
    all_files = os.listdir(INPUT_DIR)
    fb2_files = natsorted([f for f in all_files if f.lower().endswith('.fb2')])

    if not fb2_files:
        print(f"❌ В папке {INPUT_DIR} не найдено .fb2 файлов! Закинь туда книги.")
        return

    print(f"📚 Найдено книг для озвучки: {len(fb2_files)}")

    for book_filename in fb2_files:
        book_title = book_filename.rsplit('.', 1)[0]
        print(f"\n🚀 НАЧИНАЕМ РАБОТУ НАД: {book_title}")

        # Создаем папку для кусков текущей книги
        book_chunks_dir = os.path.join(SAVE_BASE_DIR, f"{book_title}_chunks")
        os.makedirs(book_chunks_dir, exist_ok=True)

        # Путь для итогового файла книги
        final_audio_path = os.path.join(SAVE_BASE_DIR, f"{book_title}_FULL.mp3")

        # Если итоговый файл уже собран, значит книга готова — пропускаем её целиком!
        if os.path.exists(final_audio_path):
            print(f"⏭️ Книга уже полностью озвучена и склеена. Идем дальше!")
            continue

        book_path = os.path.join(INPUT_DIR, book_filename)

        # 1. Достаем текст
        full_text = get_text_from_fb2(book_path)

        # 2. Бьем на куски
        chunks = chunk_text(full_text)

        # 3. Озвучиваем с защитой от сбоев
        await generate_bulletproof_audio(chunks, book_chunks_dir)

        # 4. Склеиваем
        success = merge_audio_for_book(book_chunks_dir, final_audio_path)

        if success:
            print(f"🎉 Готово! Единая книга собрана и лежит тут: {final_audio_path}")

    print("\n🏆 ВСЕ КНИГИ ИЗ ПАПКИ УСПЕШНО ОБРАБОТАНЫ!")

# Запускаем скрипт
await process_all_books()

Mounted at /content/drive
📁 Исходные книги (fb2) ищем в: /content
📁 Готовые аудиокниги будут тут: /content/drive/MyDrive/Astrology_Audiobooks
----------------------------------------
📚 Найдено книг для озвучки: 10

🚀 НАЧИНАЕМ РАБОТУ НАД: Tom 1. Vvedenie v astrologiyu
⏭️ Книга уже полностью озвучена и склеена. Идем дальше!

🚀 НАЧИНАЕМ РАБОТУ НАД: Tom 2. Gradusologiya
⏭️ Книга уже полностью озвучена и склеена. Идем дальше!

🚀 НАЧИНАЕМ РАБОТУ НАД: Tom 3. Domologiya
⏭️ Книга уже полностью озвучена и склеена. Идем дальше!

🚀 НАЧИНАЕМ РАБОТУ НАД: Tom 4. Planetologiya chast I. Solnce i Luna
⏭️ Книга уже полностью озвучена и склеена. Идем дальше!

🚀 НАЧИНАЕМ РАБОТУ НАД: Tom 6. Planetologiya chast III. Saturn Uran Neptun
⏭️ Книга уже полностью озвучена и склеена. Идем дальше!

🚀 НАЧИНАЕМ РАБОТУ НАД: Tom 8. Aspektologiya chast I. Teoriya Solnce Luna Merkuriy
⏭️ Книга уже полностью озвучена и склеена. Идем дальше!

🚀 НАЧИНАЕМ РАБОТУ НАД: Tom_7_Planetologiya_chast_IV_Pluton_Hiron_Prozerpina_Lunnye